# Etap 8 — Demonstracja PySpark: czyszczenie i normalizacja przejazdów (E1)

Notebook realizuje odpowiednik sub-joba **E1** z etapu 7 (MapReduce) w PySpark.

**Wejście:** `/processed/citibike/202506-citibike-tripdata_*.csv` (5 partycji, ~4,76 mln rekordów Citi Bike, czerwiec 2025)  
**Lookup:** `/raw/station_to_borough/2026-04-21/stations_2026-04-21.csv` (100 KB, broadcast)  
**Wyjście:** `/processed/T1_spark/` (czyste rekordy w schemacie T1)

**Reguły** (zgodnie z raportem etap 4 i etap 7):
- R1.1: data = pierwsze 10 znaków `started_at`
- R1.2: czas trwania w minutach = (`ended_at` − `started_at`) / 60
- R1.3: dzielnica startowa = lookup po `start_station_id`, fallback `Other`

**Filtry walidacyjne:**
- F1.1: czas trwania ≥ 1 min
- F1.2: czas trwania ≤ 1440 min (24 h)
- F1.3: niepuste `start_station_id`

Każdy odrzucony rekord jest zliczany — analog do counterów Hadoopa z etapu 7.

## 1. Inicjalizacja SparkSession (tryb YARN)

In [1]:
import time
from datetime import datetime
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

TS = datetime.now().strftime("%Y%m%dT%H%M%S")

spark = (SparkSession.builder
         .master("yarn")
         .appName(f"E1-clean-spark-{TS}")
         .config("spark.executor.memory", "512m")
         .config("spark.executor.cores", "1")
         .config("spark.executor.instances", "2")
         .config("spark.driver.memory", "512m")
         .getOrCreate())

print("Spark version:    ", spark.version)
print("Master:           ", spark.sparkContext.master)
print("Application ID:   ", spark.sparkContext.applicationId)
print("Run timestamp:    ", TS)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/10 19:51:28 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/05/10 19:51:42 WARN Client: Neither spark.yarn.jars nor spark.yarn.archive is set, falling back to uploading libraries under SPARK_HOME.


Spark version:     3.4.0
Master:            yarn
Application ID:    application_1778406640667_0003
Run timestamp:     20260510T195118


## 2. Wczytanie danych wejściowych

5 partycji CSV Citi Bike (czerwiec 2025) wczytywanych równolegle z HDFS jako jeden DataFrame.

In [2]:
INPUT_PATH = "/processed/citibike/202506-citibike-tripdata_*.csv"

df_raw = (spark.read
          .option("header", True)
          .option("escape", '"')
          .csv(INPUT_PATH))

total_input = df_raw.count()
print(f"Total input records: {total_input:,}")
print()
print("Schema:")
df_raw.printSchema()

Total input records: 4,759,345

Schema:
root
 |-- ride_id: string (nullable = true)
 |-- rideable_type: string (nullable = true)
 |-- started_at: string (nullable = true)
 |-- ended_at: string (nullable = true)
 |-- start_station_name: string (nullable = true)
 |-- start_station_id: string (nullable = true)
 |-- end_station_name: string (nullable = true)
 |-- end_station_id: string (nullable = true)
 |-- start_lat: string (nullable = true)
 |-- start_lng: string (nullable = true)
 |-- end_lat: string (nullable = true)
 |-- end_lng: string (nullable = true)
 |-- member_casual: string (nullable = true)



## 3. Wczytanie tablicy stacji (broadcast lookup)

Tablica `stations.csv` (~100 KB) jest mała na tyle, że Spark użyje jej jako broadcast variable w fazie join. Jest to rozwiązanie analogiczne do `DistributedCache` z etapu 7. Każdy executor otrzyma kopię tablicy w pamięci, co pozwoli na wykonywanie odczytu w czasie stałym.

In [3]:
STATIONS_PATH = "/raw/station_to_borough/2026-04-21/stations_2026-04-21.csv"

stations_full = (spark.read
                 .option("header", True)
                 .csv(STATIONS_PATH))

print("Stations schema:")
stations_full.printSchema()
print()
print("Sample rows:")
stations_full.show(3, truncate=False)

stations_lookup = stations_full.select(
    F.col("start_station_id").alias("station_id"),
    F.col("borough"),
)

print(f"Stations loaded: {stations_lookup.count():,}")

Stations schema:
root
 |-- start_station_id: string (nullable = true)
 |-- station_name: string (nullable = true)
 |-- latitude: string (nullable = true)
 |-- longitude: string (nullable = true)
 |-- borough: string (nullable = true)


Sample rows:


+----------------+--------------------+---------+----------+---------+
|start_station_id|station_name        |latitude |longitude |borough  |
+----------------+--------------------+---------+----------+---------+
|3169.07         |53 St & 4 Ave       |40.644862|-74.014531|Brooklyn |
|6303.02         |1 Ave & E 42 St     |40.748758|-73.970127|Manhattan|
|5279.03         |Centre St & Worth St|40.714948|-74.002345|Manhattan|
+----------------+--------------------+---------+----------+---------+
only showing top 3 rows



Stations loaded: 2,168


## 4. Pipeline transformacji — reguły R1 i filtry F1

Każdy rekord otrzymuje pole `decision` z jedną z czterech etykiet:
- `kept` — przechodzi wszystkie filtry, trafi do T1
- `dropped_no_station` — F1.3 (puste `start_station_id`)
- `dropped_too_short` — F1.1 (< 1 min)
- `dropped_too_long` — F1.2 (> 1440 min)

Tagowanie odbywa się podczas jednego przejścia (ewaluacja leniwa — żadna akcja nie jest wykonywana w tej komórce).

In [4]:
df_decided = (df_raw
    # R1.1: data = pierwsze 10 znaków started_at
    .withColumn("date", F.substring("started_at", 1, 10))
    # R1.2: czas trwania w minutach
    # explicit format z opcjonalną częścią milisekund [.SSS] — Citi Bike CSV
    # zawiera mieszane wartości: część bez ms ("yyyy-MM-dd HH:mm:ss"),
    # część z ms ("yyyy-MM-dd HH:mm:ss.SSS"). Domyślny parser Sparka 3.x
    # nie obsługuje takiej różnorodności, stąd jawny wzorzec
    .withColumn(
        "duration_min",
        (F.unix_timestamp("ended_at", "yyyy-MM-dd HH:mm:ss[.SSS]")
         - F.unix_timestamp("started_at", "yyyy-MM-dd HH:mm:ss[.SSS]")) / 60.0
    )
    # R1.3: lookup borough — broadcast left join
    .join(
        F.broadcast(stations_lookup),
        df_raw["start_station_id"] == stations_lookup["station_id"],
        how="left",
    )
    .withColumn("borough", F.coalesce(F.col("borough"), F.lit("Other")))
    # Tagowanie decyzji wg filtrów F1.x (kolejność = priorytet odrzuceń)
    .withColumn(
        "decision",
        F.when(
            F.col("start_station_id").isNull() | (F.col("start_station_id") == ""),
            F.lit("dropped_no_station"),
        ).when(
            F.col("duration_min") < 1.0,
            F.lit("dropped_too_short"),
        ).when(
            F.col("duration_min") > 1440.0,
            F.lit("dropped_too_long"),
        ).otherwise(F.lit("kept")),
    )
)

print("Pipeline zdefiniowany (lazy — żadna akcja jeszcze nie wykonana).")
df_decided.printSchema()

Pipeline zdefiniowany (lazy — żadna akcja jeszcze nie wykonana).
root
 |-- ride_id: string (nullable = true)
 |-- rideable_type: string (nullable = true)
 |-- started_at: string (nullable = true)
 |-- ended_at: string (nullable = true)
 |-- start_station_name: string (nullable = true)
 |-- start_station_id: string (nullable = true)
 |-- end_station_name: string (nullable = true)
 |-- end_station_id: string (nullable = true)
 |-- start_lat: string (nullable = true)
 |-- start_lng: string (nullable = true)
 |-- end_lat: string (nullable = true)
 |-- end_lng: string (nullable = true)
 |-- member_casual: string (nullable = true)
 |-- date: string (nullable = true)
 |-- duration_min: double (nullable = true)
 |-- station_id: string (nullable = true)
 |-- borough: string (nullable = false)
 |-- decision: string (nullable = false)



## 5. Liczniki — analog counterów Hadoopa

W etapie 7 (MapReduce) statystyki filtrów były gromadzone przez Hadoop Counters — wbudowany w framework mechanizm globalnego zliczania zdarzeń z poziomu mappera, agregowany przez framework po wszystkich węzłach.

W Sparku odpowiednikiem jest agregacja po polu kategoryzującym (`groupBy("decision").count()`) — w jednym przejściu przez dane otrzymujemy wszystkie liczniki naraz.

Wartości referencyjne pochodzą z dziennika etapu 7.

In [5]:
print("Liczenie statystyk filtrów (single-pass agg)...")
t_count_start = time.time()

counts_row = df_decided.groupBy("decision").count().collect()

t_count_end = time.time()

counters_spark = {row["decision"]: row["count"] for row in counts_row}
for key in ["dropped_no_station", "dropped_too_short", "dropped_too_long", "kept"]:
    counters_spark.setdefault(key, 0)

counters_mr = {
    "dropped_no_station": 1992,
    "dropped_too_short": 0,
    "dropped_too_long": 1091,
    "kept": 4756262,
}

print(f"\nCount duration: {(t_count_end - t_count_start):.1f}s\n")
print("Liczniki — porównanie Spark (etap 8) vs MR (etap 7):")
print(f"{'Counter':<25} {'Spark':>12} {'MR':>12} {'Diff':>10}")
print("-" * 62)
for key in ["dropped_no_station", "dropped_too_short", "dropped_too_long", "kept"]:
    s = counters_spark[key]
    m = counters_mr[key]
    diff = s - m
    diff_str = f"{diff:+,}" if diff != 0 else "0"
    print(f"{key:<25} {s:>12,} {m:>12,} {diff_str:>10}")

Liczenie statystyk filtrów (single-pass agg)...



Count duration: 75.5s

Liczniki — porównanie Spark (etap 8) vs MR (etap 7):
Counter                          Spark           MR       Diff
--------------------------------------------------------------
dropped_no_station               1,992        1,992          0
dropped_too_short                    0            0          0
dropped_too_long                 1,091        1,091          0
kept                         4,756,262    4,756,262          0


## 6. Materializacja `T1_spark` i pomiar czasu

Wybieramy rekordy z `decision == "kept"`, przekształcamy do schematu T1 (zgodnego z etap 7) i zapisujemy na HDFS pod ścieżką `/processed/T1_spark/`.

Format wyjściowy: TSV (tab-separated) bez nagłówka — analogicznie do `TextOutputFormat` w etapie 7, co pozwoli na bezpośrednie porównanie outputów.

In [6]:
OUTPUT_PATH = "/processed/T1_spark"

df_kept = (df_decided
    .filter(F.col("decision") == "kept")
    .select(
        F.col("ride_id"),
        F.col("date"),
        F.col("borough"),
        F.col("rideable_type"),
        F.col("member_casual"),
        F.format_string("%.2f", F.col("duration_min")).alias("ride_duration_min"),
        F.col("start_station_id"),
    )
)

print(f"Materializing to {OUTPUT_PATH} ...")
t_write_start = time.time()

(df_kept.write
        .mode("overwrite")
        .option("delimiter", "\t")
        .csv(OUTPUT_PATH))

t_write_end = time.time()
write_duration_ms = int((t_write_end - t_write_start) * 1000)
total_duration_ms = int((t_write_end - t_count_start) * 1000)

print(f"\nWrite duration:  {write_duration_ms:>7} ms ({write_duration_ms/1000:.1f}s)")
print(f"Total (count + write): {total_duration_ms:>7} ms ({total_duration_ms/1000:.1f}s)")
print(f"\nReference (etap 7, MR E1): 102 412 ms")

Materializing to /processed/T1_spark ...



Write duration:   135597 ms (135.6s)
Total (count + write):  211473 ms (211.5s)

Reference (etap 7, MR E1): 102 412 ms


## 7. Zapis dziennika wykonania

Dziennik w formacie zgodnym z etapem 7. Plik zapisywany jest w `/tmp/` wewnątrz kontenera `jupyter-lab`.

In [7]:
log_lines = [
    f"=== E1-spark start: {datetime.now().isoformat()} ===",
    f"Application ID: {spark.sparkContext.applicationId}",
    f"Input:    {INPUT_PATH}",
    f"Stations: {STATIONS_PATH}",
    f"Output:   {OUTPUT_PATH}",
    "",
    "Counters:",
    f"    dropped_no_station={counters_spark['dropped_no_station']}",
    f"    dropped_too_short={counters_spark['dropped_too_short']}",
    f"    dropped_too_long={counters_spark['dropped_too_long']}",
    f"    kept={counters_spark['kept']}",
    "",
    f"Count phase duration: {int((t_count_end - t_count_start) * 1000)} ms",
    f"Write phase duration: {write_duration_ms} ms",
    f"Total duration:       {total_duration_ms} ms",
    "",
    f"=== E1-spark done: {datetime.now().isoformat()}, duration={total_duration_ms} ms ===",
    "",
]
log_content = "\n".join(log_lines)

LOG_PATH_LOCAL = f"/tmp/{TS}_E1_spark.log"
with open(LOG_PATH_LOCAL, "w") as f:
    f.write(log_content)

print(f"Log saved to: {LOG_PATH_LOCAL} (inside jupyter-lab container)")
print()
print("Aby skopiować log na host:")
print(f"    docker cp jupyter-lab:{LOG_PATH_LOCAL} logs/spark/")
print()
print("--- log content ---")
print(log_content)

Log saved to: /tmp/20260510T195118_E1_spark.log (inside jupyter-lab container)

Aby skopiować log na host:
    docker cp jupyter-lab:/tmp/20260510T195118_E1_spark.log logs/spark/

--- log content ---
=== E1-spark start: 2026-05-10T19:58:22.057528 ===
Application ID: application_1778406640667_0003
Input:    /processed/citibike/202506-citibike-tripdata_*.csv
Stations: /raw/station_to_borough/2026-04-21/stations_2026-04-21.csv
Output:   /processed/T1_spark

Counters:
    dropped_no_station=1992
    dropped_too_short=0
    dropped_too_long=1091
    kept=4756262

Count phase duration: 75466 ms
Write phase duration: 135597 ms
Total duration:       211473 ms

=== E1-spark done: 2026-05-10T19:58:22.072383, duration=211473 ms ===



## 8. Weryfikacja zgodności z etapem 7

Wczytujemy `/processed/T1/` (output sub-joba E1 z etapu 7, MapReduce) i porównujemy liczbę wierszy z naszym `/processed/T1_spark/`.

In [8]:
df_t1_mr = (spark.read
            .option("delimiter", "\t")
            .option("header", False)
            .csv("/processed/T1"))

df_t1_spark = (spark.read
               .option("delimiter", "\t")
               .option("header", False)
               .csv("/processed/T1_spark"))

count_mr = df_t1_mr.count()
count_spark = df_t1_spark.count()

print(f"T1       (MapReduce, etap 7): {count_mr:>10,} wierszy")
print(f"T1_spark (PySpark,   etap 8): {count_spark:>10,} wierszy")
print(f"Różnica:                      {count_spark - count_mr:>+10,}")
print()
if count_spark == count_mr:
    print("Liczby wierszy IDENTYCZNE — implementacja PySpark zgodna z MR.")
else:
    print("Różnica niezerowa — przyczyną mogą być np. różnice w obsłudze edge case'ów")
    print("(np. F.unix_timestamp inaczej traktuje strefy czasowe niż DateUtils z etapu 7).")

T1       (MapReduce, etap 7):  4,756,262 wierszy
T1_spark (PySpark,   etap 8):  4,756,262 wierszy
Różnica:                              +0

Liczby wierszy IDENTYCZNE — implementacja PySpark zgodna z MR.


## 9. Zamknięcie sesji

In [9]:
spark.stop()
print("Spark session stopped.")

Spark session stopped.
